# Backblaze Hard Drive Failure Prediction — EDA


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path(".").resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Imports OK")


In [ ]:
DATA_DIR   = PROJECT_DIR / "Datasets" / "Backblaze" / "backblaze_data"

TRAIN_RAW  = DATA_DIR / "train_set.csv"
VAL_RAW    = DATA_DIR / "val_set.csv"
VAL_LABELS = DATA_DIR / "val_label.csv"
TEST_RAW   = DATA_DIR / "test_set.csv"
TEST_IDS   = DATA_DIR / "test_serial_number_id.csv"

print(f"Train: {TRAIN_RAW.exists()}  Val: {VAL_RAW.exists()}  Test: {TEST_RAW.exists()}")


---
## 1. Data Cleaning

- Drop columns with >80 % missing values
- Drop constant and duplicate columns
- Keep raw SMART metrics, drop redundant normalized counterparts
- Impute remaining NaNs fitted on train only

In [ ]:
print("Loading raw CSVs...")
train_raw = pd.read_csv(TRAIN_RAW, parse_dates=["date"], low_memory=False)
val_raw   = pd.read_csv(VAL_RAW,   parse_dates=["date"], low_memory=False)
print(f"Train raw shape: {train_raw.shape}")
print(f"Val raw shape  : {val_raw.shape}")
print(f"Failure rate in train: {train_raw['failure'].mean():.4%}")
test_raw  = pd.read_csv(TEST_RAW,  parse_dates=["date"], low_memory=False)
print(f"Test raw shape : {test_raw.shape}")


In [ ]:
# ── Raw Data Overview ────────────────────────────────────────────────────────
print("=" * 60)
print("  Backblaze Raw Data Overview")
print("=" * 60)

for split_name, df in [("Train", train_raw), ("Val", val_raw), ("Test", test_raw)]:
    disks        = df["serial_number"].nunique()
    date_range   = df.groupby("serial_number")["date"].agg(["min", "max"])
    time_span    = (date_range["max"] - date_range["min"]).dt.days + 1
    print(f"\n── {split_name} ──")
    print(f"  Rows              : {len(df):,}")
    print(f"  Columns           : {df.shape[1]}")
    print(f"  Unique disks      : {disks:,}")
    print(f"  Date range        : {df['date'].min().date()} ~ {df['date'].max().date()}")
    print(f"  Avg time span/disk: {time_span.mean():.1f} days")
    print(f"  Med time span/disk: {time_span.median():.1f} days")
    print(f"  Min time span/disk: {time_span.min()} days")
    print(f"  Max time span/disk: {time_span.max()} days")

# ── Train: failed vs never-failed ─────────────────────────────────────────────
print("\n── Train failure rate ──")
failed_disks = train_raw.groupby("serial_number")["failure"].max()
print(f"  Never-failed disks: {(failed_disks == 0).sum():,}  ({100*(failed_disks==0).mean():.1f}%)")
print(f"  Failed disks      : {(failed_disks == 1).sum():,}  ({100*(failed_disks==1).mean():.1f}%)")

# ── Train: label distribution derived from RUL ────────────────────────────────
# Label rules (same as generate_labels in backblaze_feature_engineering):
#   rul_days < 10  -> label 2  (imminent failure)
#   rul_days < 20  -> label 1  (near failure)
#   otherwise      -> label 0  (normal / censored, rul=9999)
print("\n── Train label distribution (RUL-based, row level) ──")
print("  Rules: rul<10 -> 2 (imminent),  10<=rul<20 -> 1 (near),  else -> 0 (normal)")

_tr = train_raw[["serial_number", "date", "failure"]].copy()
_fail_date = (
    _tr[_tr["failure"] == 1]
    .groupby("serial_number")["date"].min()
    .rename("failure_date")
)
_tr = _tr.join(_fail_date, on="serial_number")
_tr["rul_days"] = (_tr["failure_date"] - _tr["date"]).dt.days.fillna(9999).clip(lower=0).astype(int)
_tr["label"] = 0
_tr.loc[_tr["rul_days"] < 20, "label"] = 1
_tr.loc[_tr["rul_days"] < 10, "label"] = 2

_lbl_name = {0: "normal   (rul>=20 or censored)", 1: "near     (10<=rul<20)", 2: "imminent (rul<10)"}
_row_dist = _tr["label"].value_counts().sort_index()
for lbl, cnt in _row_dist.items():
    print(f"  label {lbl}  {_lbl_name[lbl]}: {cnt:>10,} rows  ({100*cnt/len(_tr):.2f}%)")
print(f"  Total rows: {len(_tr):,}")

print("\n── Train label distribution (per disk -- worst label per disk) ──")
_disk_worst = _tr.groupby("serial_number")["label"].max()
_disk_dist  = _disk_worst.value_counts().sort_index()
for lbl, cnt in _disk_dist.items():
    print(f"  label {lbl}  {_lbl_name[lbl]}: {cnt:>6,} disks  ({100*cnt/_disk_dist.sum():.1f}%)")

# ── Val: failure rate + label distribution (official) ───────────────────────────
val_labels_df = pd.read_csv(VAL_LABELS)
_vn = (val_labels_df["label"] == 0).sum()
_vf = (val_labels_df["label"] >  0).sum()
_vt = len(val_labels_df)
print("\n── Val failure rate (official labels) ──")
print(f"  Never-failed disks (label=0): {_vn:,}  ({100*_vn/_vt:.1f}%)")
print(f"  Failed disks       (label>0): {_vf:,}  ({100*_vf/_vt:.1f}%)")

print("\n── Val label distribution (official) ──")
print(val_labels_df["label"].value_counts().sort_index().rename("count").to_frame().assign(
    pct=lambda d: (100 * d["count"] / d["count"].sum()).round(1)
).to_string())

# ── Test disks ────────────────────────────────────────────────────────────────
print("\n── Test disks ──")
test_ids_df = pd.read_csv(TEST_IDS)
print(f"  Test disks (from test_serial_number_id): {len(test_ids_df):,}")
